# BSV PDF Parser

Parses a single Bulletin de Santé du Végétal (BSV) PDF into section records.

Each record contains bulletin metadata (`bulletin_id`, `url`, `crop`, `year`, `date`)
and section content (`section_title`, `section_index`, `text`, `images`, `tables`, `pages`).



In [ ]:
# Resolve paths from the repository root so notebooks work regardless of the kernel CWD.
from pathlib import Path
import os

def _find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "manifests" / "corpus_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the PhytoPairs repository root (expected data/manifests/corpus_manifest.csv)."
    )

PROJECT_ROOT = _find_project_root()
BSV_ROOT = PROJECT_ROOT / "3.Dataset Construction"
FINETUNING_ROOT = PROJECT_ROOT / "4.Finetuning"
CASE_STUDY_ROOT = PROJECT_ROOT / "5.BBCH case study"
RESULTS_RUNS_ROOT = PROJECT_ROOT / "results" / "runs" / "section_4"
DEFAULT_HF_CACHE = os.environ.get(
    "HF_HOME", str(Path.home() / ".cache" / "huggingface")
)


## 1. Install dependencies

In [1]:
import sys
!{sys.executable} -m pip install pymupdf pdfplumber --break-system-packages -q
print("Done")


Done


## 2. Imports

In [2]:
import os, re, shutil
from pathlib import Path
import fitz          # pymupdf
import pdfplumber
from PIL import Image as PILImage
import io


## 3. Constants

Defines constants: `KNOWN_CROPS` list for filename-based crop detection, `SIZE_THRESHOLD=10`, `LEFT_THRESHOLD=15`, `MAX_HEADER_LEN=80`, `SKIP_PAGES={1}`, `MIN_IMAGE_BYTES=5000`, footer patterns, and bad header patterns (sidebar labels, single lowercase words, decorative elements).



In [3]:
KNOWN_CROPS = [
    "vigne","ble","mais","colza","tournesol","orge","soja","pois",
    "luzerne","lin","betterave","pomme","poire","peche","cerise",
    "fraise","framboise","carotte","tomate","laitue","chou","poireau",
    "oignon","ail","melon","concombre","courgette","tabac","houblon",
    "noix","noisette","lavande","chanvre","sorgho","triticale","avoine",
    "prune","abricot","kiwi","cassis","groseille","endive","mache",
    "epinard","poivron","aubergine","asperge","artichaut","haricot",
]

SIZE_THRESHOLD  = 10   # font size (pt) threshold for section headers
LEFT_THRESHOLD  = 15   # min x position to exclude left-edge artifacts
MAX_HEADER_LEN  = 80   # max chars for a section title
SKIP_PAGES      = {1}  # skip cover page
MIN_IMAGE_BYTES = 5000 # filter logos and tiny icons
MIN_IMAGE_AREA  = 200_000

FOOTER_PATTERNS = [
    re.compile(r'bulletin de sant', re.I),
    re.compile(r'^\d+/\d+\s*$'),
    re.compile(r'^page\s+\d+', re.I),
]

BAD_HEADER_PATTERNS = [
    re.compile(r'consultez', re.I),
    re.compile(r'guide de l.observateur', re.I),
    re.compile(r'^[«»\s]+$'),
    re.compile(r'^n°\d+', re.I),
    re.compile(r'^edition\b', re.I),
    re.compile(r'^\d{2}/\d{2}/\d{4}$'),
    re.compile(r'crédit photo', re.I),
    re.compile(r'^ce qu.il faut retenir', re.I),
    re.compile(r'^[a-zàâäéèêëîïôùûüç\-]+$'),
    re.compile(r'^\d{1,2}\s+\w+\s+\d{4}', re.I),   # date strings like "16 mai 2017 N°06"
]

def is_footer(text):  return any(p.search(text) for p in FOOTER_PATTERNS)
def is_bad_header(t): return any(p.search(t)    for p in BAD_HEADER_PATTERNS)


## 4. Metadata extraction

Defines `extract_metadata`: extracts `bulletin_id`, `crop`, `date`, `year`, `region_code` from the PDF filename and URL using regex for date patterns and a keyword match against `KNOWN_CROPS`.



In [4]:
def extract_metadata(pdf_path, url=None):
    if url:
        name = Path(url).stem.lower()
    else:
        name = Path(pdf_path).stem.lower()

    meta = {"bulletin_id": Path(url).stem if url else Path(pdf_path).stem, "url": url or ""}

    # Year
    meta["year"] = None
    if url:
        m = re.search(r'/(20\d{2})/', url)
        if m: meta["year"] = m.group(1)
    if not meta["year"]:
        m = re.search(r'(20\d{2})', name)
        if m: meta["year"] = m.group(1)

    # Date
    meta["date"] = None

    # YYYYMMDD — only match if starts with 20
    m = re.search(r'(20\d{6})', name)
    if m:
        d = m.group(1)
        meta["date"] = f"{d[:4]}-{d[4:6]}-{d[6:8]}"

    # DDMMYYYY — 8 digits where last 4 form a valid year
    if not meta["date"]:
        m = re.search(r'(\d{2})(\d{2})(20\d{2})', name)
        if m:
            day, month, year = m.group(1), m.group(2), m.group(3)
            if 1 <= int(day) <= 31 and 1 <= int(month) <= 12:
                meta["date"] = f"{year}-{month}-{day}"

    # YYYY-MM-DD or YYYY_MM_DD
    if not meta["date"]:
        m = re.search(r'(20\d{2})[_-](\d{2})[_-](\d{2})', name)
        if m:
            year, month, day = m.group(1), m.group(2), m.group(3)
            meta["date"] = f"{year}-{month}-{day}"

    # DD-MM-YY or DD_MM_YY
    if not meta["date"]:
        m = re.search(r'(?<!\d)(\d{2})[_-](\d{2})[_-](\d{2,4})(?!\d)', name)
        if m:
            day, month, year = m.group(1), m.group(2), m.group(3)
            if len(year) == 2:
                year = "20" + year
            if 1 <= int(day) <= 31 and 1 <= int(month) <= 12 and 2000 <= int(year) <= 2030:
                meta["date"] = f"{year}-{month}-{day}"

    # Crop
    CROP_CATEGORIES = {
        "grandes_cultures": "Grandes Cultures",
        "maraichage":       "Maraîchage",
        "viti":             "Viticulture",
        "vigne":            "Viticulture",
        "arboriculture":    "Arboriculture",
        "arbo":             "Arboriculture",
        "legume":           "Légumes",
        "lin":              "Lin",
        "colza":            "Colza",
        "tournesol":        "Tournesol",
        "pomme_de_terre":   "Pomme de terre",
        "pommedeterre":     "Pomme de terre",
        "pdt":              "Pomme de terre",
        "jevi":             "JEVI",
        "zna":              "ZNA",
        "canne":            "Canne à sucre",
        "olivier":          "Olivier",
        "cereale":          "Céréales",
        "mais":             "Maïs",
        "tabac":            "Tabac",
    }
    meta["crop"] = next((CROP_CATEGORIES[k] for k in CROP_CATEGORIES if k in name), None)

    # Region code
    meta["region_code"] = None
    if url:
        m = re.search(r'/pdf/(Q\d+)/', url)
        if m: meta["region_code"] = m.group(1)

    return meta


## 5. Parse PDF structure (text positions, images)

Defines `parse_structure`: counts xref occurrences to identify recurring logos (filtered if appearing on more than 2 pages). Computes a per-document dynamic size threshold: finds the max bold size on content pages, excludes rare very-large sizes (appearing ≤ 2 times and more than 30% larger than the next size), then uses 85% of that as the threshold. Section headers are bold spans above this threshold, not in the bad header list, left position ≥ 15px. Skips page 1 for bulletins longer than 2 pages. Merges candidate header spans within 8px vertically on the same page.



In [5]:
def parse_structure(pdf_path):
    from collections import Counter

    doc = fitz.open(pdf_path)
    total_pages = doc.page_count
    all_text_els, images_raw_all, xref_page_count = [], [], {}

    for page_num, page in enumerate(doc, 1):
        if page_num == 1 and total_pages > 2: continue
        seen_xrefs = set()
        for img in page.get_images(full=True):
            xref = img[0]
            if xref not in seen_xrefs:
                xref_page_count[xref] = xref_page_count.get(xref, 0) + 1
                seen_xrefs.add(xref)
            rects = page.get_image_rects(xref)
            if not rects:
                continue
            top = rects[0].y0
            images_raw_all.append({"page": page_num, "top": top, "xref": xref})

        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0:
                continue
            for line in block["lines"]:
                for span in line["spans"]:
                    text = span["text"].strip()
                    if not text or is_footer(text):
                        continue
                    x0    = span["bbox"][0]
                    y0    = span["bbox"][1]
                    size  = span["size"]
                    bold  = bool(span["flags"] & 16) or "Bold" in span.get("font", "")
                    color = span.get("color", 0)  # CHANGED: store text color
                    all_text_els.append({"page": page_num, "top": y0, "left": x0,
                                         "size": size, "text": text, "bold": bold,
                                         "color": color})
    doc.close()

    # Filter recurring logos and deduplicate xrefs
    seen, images_raw = set(), []
    for img in images_raw_all:
        if xref_page_count.get(img["xref"], 0) <= 2 and img["xref"] not in seen:
            seen.add(img["xref"])
            images_raw.append(img)

    # Skip page 1 only for longer bulletins
    skip_pages = {1} if total_pages > 2 else set()

    # Exclude decorative cover-page titles (rare very-large sizes)
    content_bold_sizes = [
        el["size"] for el in all_text_els
        if el["page"] not in skip_pages and el["bold"] and el["size"] >= 10
    ]
    excluded_sizes = set()
    if content_bold_sizes:
        size_counts  = Counter(content_bold_sizes)
        unique_sizes = sorted(size_counts.keys(), reverse=True)
        for i, size in enumerate(unique_sizes):
            if (i + 1 < len(unique_sizes)
                    and size_counts[size] <= 2
                    and size > 1.3 * unique_sizes[i + 1]):
                excluded_sizes.add(size)
            else:
                break

    # Collect ALL bold candidates (no single size threshold)
    # Added: text must not end with punctuation (rules out bold body emphasis)
    # Added: color stored for level assignment
    PUNCT_ENDS = set('.,;!?')
    candidates = [
        {"page": el["page"], "top": el["top"], "text": el["text"],
         "size": el["size"], "color": el["color"]}
        for el in all_text_els
        if (el["page"] not in skip_pages
            and el["bold"]
            and el["size"] >= SIZE_THRESHOLD
            and el["size"] not in excluded_sizes
            and el["left"] >= LEFT_THRESHOLD
            and len(el["text"]) < MAX_HEADER_LEN
            and len(el["text"].strip()) >= 3
            and el["text"].strip()[-1] not in PUNCT_ENDS
            and not is_bad_header(el["text"])
            and not el["text"].strip() == el["text"].strip().lower())
    ]

    # CHANGED: assign section levels using color + size
   # --- Level assignment: adapted from pdf2blocs guess_structure ---

    def color_brightness(color):
        r = (color >> 16) & 0xFF
        g = (color >> 8) & 0xFF
        b = color & 0xFF
        return 0.299 * r + 0.587 * g + 0.114 * b

    for c in candidates:
        c["is_light"] = color_brightness(c["color"]) > 200

    # Step 1: adjusted size score per unique (size, is_light) type.
    # is_light gets +0.5 (equivalent to p2b's has_style boost).
    # Rare sizes get a larger boost; frequent sizes get less
    # (equivalent to p2b's nb_blocks frequency adjustment).
    size_counts = Counter(c["size"] for c in candidates)

    def _adjusted_score(size, is_light, count):
        sz = float(size)
        if is_light:
            sz += 0.5
        sz += 0.45 - min(0.45, 0.1 * float(round(count / 3)))
        return sz

    unique_types = sorted(
        set((c["size"], c["is_light"]) for c in candidates),
        key=lambda t: _adjusted_score(t[0], t[1], size_counts[t[0]]),
        reverse=True
    )
    type_to_depth = {t: i for i, t in enumerate(unique_types)}  # 0-indexed

    for c in candidates:
        c["_depth"] = type_to_depth[(c["size"], c["is_light"])]

    # Step 2: succession analysis (p2b's TITLE_OPTIMISE_LEVEL).
    # Group consecutive candidates of the same type, identical to p2b grouping
    # consecutive same-font blocks into title_blocks.
    sorted_cands = sorted(candidates, key=lambda x: (x["page"], x["top"]))
    depth_sequence = []
    prev_type = None
    for c in sorted_cands:
        ctype = (c["size"], c["is_light"])
        if ctype != prev_type:
            depth_sequence.append(type_to_depth[ctype])
            prev_type = ctype

    # For each depth d, record which depths directly precede it.
    n_depths = len(unique_types)
    preced = [[] for _ in range(n_depths)]
    for d, p in zip(depth_sequence[1:], depth_sequence[:-1]):
        if d > p and p not in preced[d]:
            preced[d].append(p)

    # Compute actual depth from precedence (p2b's title_font_depth).
    actual_depth = []
    for i in range(n_depths):
        p = preced[i]
        if len(p) > 0:
            actual_depth.append(max(actual_depth[d] for d in p) + 1)
        else:
            actual_depth.append(0 if len(actual_depth) == 0 else 1)

    # Step 3: assign final section_level (1-indexed, capped at 4).
    for c in candidates:
        d = c["_depth"]
        c["section_level"] = min(actual_depth[d] + 1, 4)
        del c["_depth"]

    # Merge spans on the same header line (within 8px vertically)
    sections_raw = []
    for c in sorted(candidates, key=lambda x: (x["page"], x["top"])):
        if (sections_raw
                and sections_raw[-1]["page"] == c["page"]
                and abs(sections_raw[-1]["top"] - c["top"]) <= 8):
            sections_raw[-1]["text"] += " " + c["text"]
        else:
            sections_raw.append(c)

    return sections_raw, images_raw, all_text_els

## 6. Extract full text per page
 Defines `extract_text`: runs pymupdf text extraction per page, strips footer lines, returns a list of page strings.

In [6]:
def extract_text(pdf_path):
    doc   = fitz.open(pdf_path)
    pages = []
    for page in doc:
        text  = page.get_text("text")
        lines = [l for l in text.splitlines() if not is_footer(l)]
        pages.append("\n".join(lines).strip())
    doc.close()
    return pages


## 7. Split text into sections

Defines `split_into_sections`: inserts page markers into the full text, then searches for each section title starting from that page's marker position to avoid false matches in cover page summaries. Handles bullet-prefixed titles (`\uf0b7`, `•`, `-`).


In [7]:
def split_into_sections(page_texts, sections_raw):
    seen, ordered_headers = set(), []
    for s in sorted(sections_raw, key=lambda x: (x["page"], x["top"])):
        key = s["text"].strip().lower()
        if key not in seen:
            seen.add(key)
            ordered_headers.append(s)

    PAGE_SEP  = "\n\n<<<PAGE_{page}>>>\n\n"
    full_text = "".join(PAGE_SEP.format(page=i+1) + pt
                        for i, pt in enumerate(page_texts))

    splits = []
    for h in ordered_headers:
        title       = h["text"].strip()
        marker      = f"<<<PAGE_{h['page']}>>>"
        marker_pos  = full_text.find(marker)
        search_from = marker_pos if marker_pos != -1 else 0

        idx = full_text.find(title, search_from)
        if idx == -1:
            for prefix in ["\uf0b7 ", "\uf0fc ", "• ", "- "]:
                idx = full_text.find(prefix + title, search_from)
                if idx != -1:
                    idx += len(prefix)
                    break
        if idx == -1:
            idx = full_text.lower().find(title.lower(), search_from)
        if idx != -1:
            splits.append((idx, title, h["page"], h["top"], h.get("section_level", 1)))

    splits.sort(key=lambda x: x[0])
    if not splits:
        return [{"section_index": 0, "section_title": "Document",
         "section_level": 1,
         "pages": list(range(1, len(page_texts) + 1)),
         "header_top": 0, "text": re.sub(r'<<<PAGE_\d+>>>', '', full_text).strip()}]

    sections = []
    for i, (pos, title, page, top, level) in enumerate(splits):
        end   = splits[i+1][0] if i+1 < len(splits) else len(full_text)
        chunk = re.sub(r'<<<PAGE_\d+>>>', '', full_text[pos:end]).strip()
        if chunk.lower().startswith(title.lower()):
            chunk = chunk[len(title):].strip()
        sections.append({"section_index": i, "section_title": title,
                  "section_level": level, "pages": [page], 
                  "header_top": top, "text": chunk})
    return sections


## 8. Extract images

Defines `extract_images` and `match_captions`. `extract_images` deduplicates by xref, skips images below `MIN_IMAGE_BYTES`, saves each to `out_img_dir`. `match_captions` assigns the nearest small text element (size ≤ 10, length < 120 chars) below each image as its caption.


In [8]:
def extract_images(pdf_path, out_img_dir, bulletin_id, images_raw):
    os.makedirs(out_img_dir, exist_ok=True)
    doc, result, seen = fitz.open(pdf_path), [], set()
    for img_info in images_raw:
        xref = img_info["xref"]
        if xref in seen:
            continue
        seen.add(xref)
        try:
            img_data = doc.extract_image(xref)
        except Exception:
            continue
        if len(img_data["image"]) < MIN_IMAGE_BYTES:
            continue
        if len(img_data["image"]) < MIN_IMAGE_BYTES:
            continue
        if img_data["width"] * img_data["height"] < MIN_IMAGE_AREA:
            continue

        if img_data["width"] / max(img_data["height"], 1) > 8:
            continue  # separator strip or footer banner

        pil_img = PILImage.open(io.BytesIO(img_data["image"])).convert("RGB")
        pixels = list(pil_img.getdata())
        total = len(pixels)
        
        # H36: skip images where more than 80% of pixels are near-white
        near_white = sum(1 for r, g, b in pixels if r > 240 and g > 240 and b > 240)
        if near_white / total > 0.80:
            continue
        
        # H37: skip images where a single color dominates more than 70%
        from collections import Counter
        quantized = pil_img.quantize(colors=8)
        color_counts = Counter(quantized.getdata())
        if max(color_counts.values()) / total > 0.70:
            continue
        new_name = f"{bulletin_id}_p{img_info['page']}_{xref}.{img_data['ext']}"
        with open(os.path.join(out_img_dir, new_name), "wb") as f:
            f.write(img_data["image"])
        result.append({"filename": new_name, "page": img_info["page"],
                        "top": img_info["top"], "caption": None})
    doc.close()
    return result


def match_captions(images, all_text_els):
    for img in images:
        candidates = sorted(
            [el for el in all_text_els
             if el["page"] == img["page"]
             and el["top"] > img["top"]
             and el["size"] <= 10
             and len(el["text"]) < 120],
            key=lambda x: x["top"]
        )
        if candidates:
            img["caption"] = candidates[0]["text"]
    return images


## 9. Extract tables

Defines `extract_tables`: runs pdfplumber on content pages. Filters out tables with fewer than 3 rows or 2 columns. Rejects layout artifacts where average cell text length exceeds 200 characters.


In [9]:
def extract_tables(pdf_path):
    result = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            if page_num in SKIP_PAGES:
                continue
            for table in page.extract_tables():
                if not table or len(table) < 3:
                    continue
                if max(len(row) for row in table) < 2:
                    continue
                flat = [c for row in table for c in row if c]
                avg_len = sum(len(c) for c in flat) / max(len(flat), 1)
                if avg_len > 200:
                    continue
                all_cells = [c for row in table for c in row]
                empty_cells = sum(1 for c in all_cells if not c or str(c).strip() == '')
                if empty_cells / max(len(all_cells), 1) > 0.5:
                    continue
                result.append({"page": page_num, "rows": table})
    return result


## 10. Assign images and tables to sections

Defines `assign_to_sections`: maps images to sections by vertical position (image goes to the section whose header is closest above it on the same page). Maps tables to sections by page number.


In [10]:
def assign_to_sections(sections, images, tables):
    page_headers = [(p, sec["header_top"], sec["section_index"])
                    for sec in sections for p in sec["pages"]]
    page_headers.sort()

    def find_section(img_page, img_top):
        above = [(top, idx) for (p, top, idx) in page_headers
                 if p == img_page and top <= img_top]
        if above: return max(above, key=lambda x: x[0])[1]
        same  = [(top, idx) for (p, top, idx) in page_headers if p == img_page]
        return min(same, key=lambda x: x[0])[1] if same else None

    sec_map = {sec["section_index"]: sec for sec in sections}
    for sec in sections:
        sec["images"] = []
        sec["tables"] = []
    for img in images:
        idx = find_section(img["page"], img.get("top", 0))
        if idx is not None and idx in sec_map:
            sec_map[idx]["images"].append(img)
    for t in tables:
        for sec in sections:
            if t["page"] in sec["pages"]:
                sec["tables"].append(t)
                break
    return sections


## 11. Main parse_pdf function

Defines `parse_pdf`: calls all the above in order, then merges any section with empty text into the next one by concatenating their titles. This handles PDFs where a section header is split across two spans more than 8px apart vertically.


In [11]:
def parse_pdf(pdf_path, out_img_dir, url=None):
    meta       = extract_metadata(pdf_path, url)
    s_raw, i_raw, text_els = parse_structure(pdf_path)
    page_texts = extract_text(pdf_path)
    sections   = split_into_sections(page_texts, s_raw)

    # Merge empty sections into the next one (handles split header spans)
    merged, i = [], 0
    while i < len(sections):
        sec = sections[i]
        if len(sec["text"].strip()) == 0 and i + 1 < len(sections):
            sections[i + 1]["section_title"] = (
                sec["section_title"] + " " + sections[i + 1]["section_title"]
            )
            i += 1
            continue
        merged.append(sec)
        i += 1
    sections = merged

    images     = extract_images(pdf_path, out_img_dir, meta["bulletin_id"], i_raw)
    images     = match_captions(images, text_els)
    tables     = extract_tables(pdf_path)
    sections   = assign_to_sections(sections, images, tables)
    return [{**meta, **sec} for sec in sections]


In [12]:
import os
print(os.getcwd())
print(os.listdir(os.getcwd()))

/projects/agrifutur/yabusaqe/BSV Sample Construction
['test_bulletin.pdf', '.ipynb_checkpoints', 'bsv_parser.ipynb', 'bsv_parser.py', 'bsv_batch.ipynb', 'bsv_pair_constructor.py', '__pycache__', 'PDF_URLS.txt', 'Previous Iterations Data', 'generate_reviewer.py', 'bsv_reviewer.html', 'bsv_heuristics_analysis.ipynb', 'bsv_download.ipynb', 'bsv_parser_subsections.ipynb', 'test_images', 'bsv_images', 'bsv_dataset.jsonl', 'processed_urls.txt', 'bsv_sample_500.jsonl', 'bsv_pairs.jsonl', 'bsv_pairs_sample_1000.jsonl', 'bsv_pairs_sample_500.jsonl']


In [13]:
import urllib.request

url = "https://rdf.codex.cati.inrae.fr/bsv/files/pdf/Q18677983/2017/alsace_vigne_no6_du_16-05-17_cle8eb3bd.pdf"
TEST_PDF = str(PROJECT_ROOT / "data" / "local" / "tests" / "test_bulletin.pdf")
os.makedirs(os.path.dirname(TEST_PDF), exist_ok=True)
dest = TEST_PDF

urllib.request.urlretrieve(url, dest)
print(f"Downloaded to: {dest}")
print(f"Size: {os.path.getsize(dest) / 1024:.1f} KB")

Downloaded to: /projects/agrifutur/yabusaqe/BSV Sample Construction/test_bulletin.pdf
Size: 805.6 KB


## 12. Test on a single PDF

In [14]:
TEST_PDF     = str(PROJECT_ROOT / "data" / "local" / "tests" / "test_bulletin.pdf")
TEST_IMG_DIR = str(PROJECT_ROOT / "data" / "local" / "tests" / "test_images")

records = parse_pdf(TEST_PDF, TEST_IMG_DIR)
print(f"Extracted {len(records)} sections\n")
for r in records:
    print(f"  [{r['section_index']}] {r['section_title']!r}")
    print(f"      pages={r['pages']}  text={len(r['text'])}chars  "
          f"images={len(r['images'])}  tables={len(r['tables'])}")
    print(f"      preview: {r['text'][:120]!r}")
    print()


/tmp/ipykernel_2158822/1531050965.py:24: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(pil_img.getdata())
/tmp/ipykernel_2158822/1531050965.py:35: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  color_counts = Counter(quantized.getdata())


Extracted 7 sections

  [0] 'Tordeuses'
      pages=[1]  text=15chars  images=0  tables=0
      preview: 'Quelques pontes'

  [1] 'Mildiou'
      pages=[1]  text=12chars  images=0  tables=0
      preview: 'Aucune tache'

  [2] 'Oïdium'
      pages=[1]  text=14chars  images=0  tables=0
      preview: 'Aucun symptôme'

  [4] 'Point météo/Phénologie'
      pages=[1]  text=886chars  images=0  tables=0
      preview: 'La hausse des températures permet \nd’amorcer une reprise plus active de \nla végétation avec en moyenne 6-7 \nfeuilles éta'

  [5] 'Piégeage'
      pages=[1]  text=191chars  images=0  tables=0
      preview: 'Le rythme des captures commence à \ndécroitre cette semaine. \nDes pontes sont identifiées dans les 2 \ndépartements à haut'

  [6] 'Analyse du risque'
      pages=[1]  text=1374chars  images=0  tables=0
      preview: 'L’activité de vol va lentement sur sa fin \net les pontes pourront encore être \ndéposées cette semaine.  \nAucune \nsituati'

  [7] 'Situation'
      pa

In [15]:
import fitz

doc = fitz.open(TEST_PDF)
print(f"Total pages: {len(doc)}\n")

for page_num, page in enumerate(doc, 1):
    print(f"--- PAGE {page_num} ---")
    for block in page.get_text("dict")["blocks"]:
        if block["type"] != 0:
            continue
        for line in block["lines"]:
            for span in line["spans"]:
                text = span["text"].strip()
                if not text:
                    continue
                bold = bool(span["flags"] & 16) or "Bold" in span.get("font", "")
                size = span["size"]
                x0   = span["bbox"][0]
                # Only show text that could plausibly be a header
                if size >= 10 and len(text) < 80:
                    print(f"  size={size:.1f} bold={bold} x0={x0:.0f}  {text!r}")
    print()

Total pages: 2

--- PAGE 1 ---
  size=30.0 bold=True x0=483  'Viticulture'
  size=16.0 bold=True x0=67  'Phénologie'
  size=11.0 bold=False x0=67  '6-7 feuilles'
  size=16.0 bold=True x0=67  'Tordeuses'
  size=11.0 bold=False x0=67  'Quelques pontes'
  size=16.0 bold=True x0=67  'Mildiou'
  size=11.0 bold=False x0=67  'Aucune tache'
  size=16.0 bold=True x0=67  'Oïdium'
  size=11.0 bold=False x0=67  'Aucun symptôme'
  size=20.0 bold=True x0=173  'Point'
  size=20.0 bold=True x0=173  'météo/Phénologie'
  size=11.0 bold=False x0=173  'La hausse des températures permet'
  size=11.0 bold=False x0=173  'd’amorcer une reprise plus active de'
  size=11.0 bold=False x0=173  'la végétation avec en moyenne 6-7'
  size=11.0 bold=False x0=173  'feuilles étalées masquant toujours un'
  size=11.0 bold=False x0=173  'énorme'
  size=11.0 bold=False x0=223  'contraste'
  size=11.0 bold=False x0=281  'au'
  size=11.0 bold=False x0=306  'sein'
  size=11.0 bold=False x0=339  'des'
  size=11.0 bold=False x

In [16]:
from collections import Counter

# Re-run parse_structure to get all_text_els
s_raw, i_raw, text_els = parse_structure(TEST_PDF)

import fitz
total_pages = fitz.open(TEST_PDF).page_count
print(f"Total pages: {total_pages}")

bold_sizes = [el["size"] for el in text_els if el["bold"] and el["size"] >= 10]
print("\nBold size distribution:")
for size, count in sorted(Counter(bold_sizes).items(), reverse=True):
    print(f"  size={size:.1f}  count={count}")

Total pages: 2

Bold size distribution:
  size=30.0  count=1
  size=20.0  count=5
  size=16.0  count=4
  size=11.0  count=7


In [17]:
TEST_URL = "https://rdf.codex.cati.inrae.fr/bsv/files/pdf/Q18677875/2019/bsv_viti_als_n02_du_14_05_2019_cle8c93cf.pdf"
result = parse_pdf(TEST_PDF, TEST_IMG_DIR, url=TEST_URL)
for s in result:
    print(f"[{s['section_index']}] {s['section_title']!r}")
    print(f"    pages={s['pages']}  text={len(s['text'])}chars  images={len(s['images'])}  tables={len(s['tables'])}")
    print(f"    preview: {s['text'][:100]!r}")
    print()

# Print metadata separately
meta_keys = ['bulletin_id', 'crop', 'date', 'year', 'region_code']
print("=== METADATA ===")
for k in meta_keys:
    print(f"  {k}: {result[0].get(k)!r}")

for s in result:
    print(f"[{s['section_index']}] {s['section_title']!r}  level={s.get('section_level', '?')}")
    print(f"    pages={s['pages']}  text={len(s['text'])}chars  images={len(s['images'])}  tables={len(s['tables'])}")
    print()

[0] 'Tordeuses'
    pages=[1]  text=15chars  images=0  tables=0
    preview: 'Quelques pontes'

[1] 'Mildiou'
    pages=[1]  text=12chars  images=0  tables=0
    preview: 'Aucune tache'

[2] 'Oïdium'
    pages=[1]  text=14chars  images=0  tables=0
    preview: 'Aucun symptôme'

[4] 'Point météo/Phénologie'
    pages=[1]  text=886chars  images=0  tables=0
    preview: 'La hausse des températures permet \nd’amorcer une reprise plus active de \nla végétation avec en moyen'

[5] 'Piégeage'
    pages=[1]  text=191chars  images=0  tables=0
    preview: 'Le rythme des captures commence à \ndécroitre cette semaine. \nDes pontes sont identifiées dans les 2 '

[6] 'Analyse du risque'
    pages=[1]  text=1374chars  images=0  tables=0
    preview: 'L’activité de vol va lentement sur sa fin \net les pontes pourront encore être \ndéposées cette semain'

[7] 'Situation'
    pages=[2]  text=1076chars  images=1  tables=0
    preview: 'La situation reste inchangée par rapport à la \nsemaine \ndernière. 

/tmp/ipykernel_2158822/1531050965.py:24: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(pil_img.getdata())
/tmp/ipykernel_2158822/1531050965.py:35: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  color_counts = Counter(quantized.getdata())


In [18]:
import os
PDF_DIR = str(PROJECT_ROOT / "data" / "local" / "pdfs")
viti = [f for f in os.listdir(PDF_DIR) if 'viti' in f.lower() or 'vigne' in f.lower()]
print(f"Found {len(viti)} viticulture PDFs")
for f in viti[:5]:
    print(f)

Found 2358 viticulture PDFs
bsv_vigne_n17_cahors_25072017_cle8db279.pdf
bsv_viti_mp_gaillac_n08_26052021_1__cle0ecee7.pdf
BSVViti_no11_05062018_cle441373.pdf
BSV_NA_VIGNE_Sud_Aquitaine_09_20200519_cle865562.pdf
bsv_viti_mp_gascogne_n11_12062019_cle82a1dd.pdf


In [19]:
TEST_PDF2 = str(PROJECT_ROOT / "data" / "local" / "pdfs" / 'BSV_NA_VIGNE_Sud_Aquitaine_09_20200519_cle865562.pdf')
result2 = parse_pdf(TEST_PDF2, TEST_IMG_DIR, url=None)
for s in result2:
    print(f"[{s['section_index']}] level={s.get('section_level','?')}  {s['section_title']!r}")
    print(f"    text={len(s['text'])}chars")
    print()

/tmp/ipykernel_2158822/1531050965.py:24: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(pil_img.getdata())


[0] level=1  'Données météorologiques de la semaine passée'
    text=1chars

[1] level=2  'Températures'
    text=481chars

[2] level=2  'Pluviométries'
    text=144chars

[3] level=1  'Etat général du vignoble'
    text=1chars

[4] level=2  'Stades phénologiques'
    text=723chars

[5] level=1  'Maladies fongiques'
    text=1chars

[6] level=2  'Mildiou'
    text=3998chars

[7] level=2  'Black rot'
    text=3398chars

[8] level=2  'Oïdium'
    text=2486chars

[9] level=1  'Ravageurs'
    text=1chars

[10] level=2  'Cicadelles vertes'
    text=42chars

[11] level=2  'Cicadelles de la Flavescence dorée'
    text=454chars

[12] level=2  'Vers de la grappe'
    text=2707chars

[13] level=2  'Protection des abeilles'
    text=2272chars

[14] level=1  'Résistances aux produits de protection des plantes'
    text=1093chars

[15] level=2  'Prochain bulletin : le mardi 26 mai'
    text=1095chars



In [20]:
import os
lorraine = [f for f in os.listdir(PDF_DIR) if 'LOR' in f or 'lorraine' in f.lower()]
print(lorraine[:10])

['20180719_LOR_BSV_Horticulture-Pepiniere_cle8ad46b.pdf', '20210512_LOR_Viticulture-n5_cle094443.pdf', '20210527_LOR_Viticulture-n7_cle0e5712.pdf', '20180905_LOR_BSV_Grandes_Cultures_cle85391c.pdf', '20180711_LOR_BSV_Grandes_Cultures_cle8b1bd7.pdf', 'lorraine_gdes_cultures_no18_du16-06-21_cle834dc3-1.pdf', 'BSV02_VITI_LOR_S19_2020_cle819d15.pdf', 'lorraine_arboriculture_no12_du_01-06-22.pdf', '20190515_LOR_BSV_Grandes_Cultures_cle861a8d.pdf', '20212107_LOR_Viticulture-n15_cle815d64.pdf']


In [21]:
TEST_PDF2 = str(PROJECT_ROOT / "data" / "local" / "pdfs" / '20210512_LOR_Viticulture-n5_cle094443.pdf')
result2 = parse_pdf(TEST_PDF2, TEST_IMG_DIR, url=None)
for s in result2:
    print(f"[{s['section_index']}] level={s.get('section_level','?')}  {s['section_title']!r}")
    print(f"    text={len(s['text'])}chars")
    print()

/tmp/ipykernel_2158822/1531050965.py:24: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(pil_img.getdata())
/tmp/ipykernel_2158822/1531050965.py:35: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  color_counts = Counter(quantized.getdata())


[0] level=2  'Stade des cultures'
    text=438chars

[1] level=2  'Données météo'
    text=1183chars

[2] level=1  'PHENOLOGIE'
    text=233chars

[3] level=2  'Situation actuelle'
    text=145chars

[4] level=2  'Analyse de risque'
    text=1450chars

[5] level=3  'Aucun risque'
    text=361chars

[7] level=1  'MILDIOU TORDEUSES'
    text=512chars

[8] level=1  'PYRALES'
    text=1236chars



In [22]:
TEST_CHARENTES = str(PROJECT_ROOT / "data" / "local" / "pdfs" / 'BSV_NA_VIGNE_CHARENTES_02_20190409_cle8ac81e.pdf')
result3 = parse_pdf(TEST_CHARENTES, TEST_IMG_DIR, url=None)
for s in result3:
    print(f"[{s['section_index']}] level={s.get('section_level','?')}  {s['section_title']!r}")
    print(f"    text={len(s['text'])}chars")
    print()

/tmp/ipykernel_2158822/1531050965.py:24: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(pil_img.getdata())
/tmp/ipykernel_2158822/1531050965.py:35: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  color_counts = Counter(quantized.getdata())


[0] level=1  'Phénologie'
    text=519chars

[1] level=1  'Climatologie'
    text=1chars

[2] level=2  'De la semaine passée'
    text=252chars

[3] level=2  'Prévisions météo'
    text=80chars

[4] level=1  'Gel, grêle'
    text=97chars

[5] level=1  'Maladies'
    text=1chars

[6] level=2  'Rappel modélisation'
    text=2569chars

[7] level=2  'Mildiou :'
    text=2744chars

[8] level=2  'Black rot'
    text=2980chars

[9] level=2  'Excoriose'
    text=842chars

[10] level=1  'Ravageurs'
    text=1chars

[11] level=2  'Tordeuses'
    text=1409chars

[12] level=2  'Mange-bourgeons'
    text=347chars

[13] level=2  'Escargots'
    text=342chars

[14] level=2  'Le Mémo de l’Observateur'
    text=2265chars



In [23]:
import fitz
from collections import Counter
doc = fitz.open(str(PROJECT_ROOT / "data" / "local" / "pdfs" / 'BSV_NA_VIGNE_CHARENTES_02_20190409_cle8ac81e.pdf'))
for page_num, page in enumerate(doc, 1):
    for block in page.get_text("dict")["blocks"]:
        if block["type"] != 0: continue
        for line in block["lines"]:
            for span in line["spans"]:
                text = span["text"].strip()
                bold = bool(span["flags"] & 16) or "Bold" in span.get("font","")
                if bold and span["size"] <= 13 and len(text) > 3:
                    print(f"p{page_num} size={span['size']:.1f} last={text[-1]!r} text={text!r}")

p1 size=7.0 last='e' text='Animateur filière'
p1 size=7.0 last='e' text='Chambre d’agriculture de'
p1 size=7.0 last='e' text='la Charente-Maritime'
p1 size=7.0 last='n' text='Directeur de publication'
p1 size=7.0 last='n' text='Supervision'
p1 size=6.0 last='e' text='Reproduction intégrale'
p1 size=6.0 last='.' text='de ce bulletin autorisée.'
p1 size=6.0 last='e' text='Reproduction partielle'
p1 size=6.0 last='n' text='autorisée avec la mention'
p1 size=6.0 last='é' text='« extrait du bulletin de santé'
p1 size=6.0 last='e' text='du végétal Nouvelle-Aquitaine'
p1 size=6.0 last='s' text='Vigne / Edition Charentes'
p1 size=6.0 last='»' text='N°2 du 09/04/2019 »'
p1 size=10.0 last='e' text='Web Alerte Vigne'
p1 size=12.0 last='9' text='09/04/2019'
p1 size=12.0 last='n' text='Edition'
p1 size=9.0 last='T' text='Recevez le Bulletin de votre choix GRATUITEMENT'
p1 size=9.0 last='r' text='en cliquant sur'
p1 size=9.0 last='V' text="Formulaire d'abonnement au BSV"
p1 size=12.0 last='e' text='

In [24]:
print(SIZE_THRESHOLD)
print(LEFT_THRESHOLD)
print(MAX_HEADER_LEN)

10
15
80


In [27]:
import fitz, io
from PIL import Image as PILImage
from collections import Counter
TEST_LEGUMES = str(PROJECT_ROOT / "data" / "local" / "pdfs" / 'BSV_no10_legumes_07_juin_2019_cle81a635.pdf')
doc = fitz.open(TEST_LEGUMES)
for page_num, page in enumerate(doc, 1):
    for img in page.get_images(full=True):
        xref = img[0]
        try:
            img_data = doc.extract_image(xref)
        except:
            continue
        pil_img = PILImage.open(io.BytesIO(img_data["image"])).convert("RGB")
        pixels = list(pil_img.getdata())
        total = len(pixels)
        near_white = sum(1 for r,g,b in pixels if r>240 and g>240 and b>240)
        quantized = pil_img.quantize(colors=8)
        color_counts = Counter(quantized.getdata())
        dominant = max(color_counts.values())
        h36 = near_white / total
        h37 = dominant / total
        verdict = "SKIP" if h36 > 0.80 or h37 > 0.70 else "KEEP"
        print(f"p{page_num} xref={xref} size={img_data['width']}x{img_data['height']} H36={h36:.2f} H37={h37:.2f} → {verdict}")

/tmp/ipykernel_2158822/678445069.py:14: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(pil_img.getdata())
/tmp/ipykernel_2158822/678445069.py:18: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  color_counts = Counter(quantized.getdata())


p1 xref=5 size=2480x3508 H36=0.88 H37=0.87 → SKIP
p2 xref=31 size=2480x3508 H36=0.98 H37=0.95 → SKIP
p2 xref=32 size=397x505 H36=0.89 H37=0.89 → SKIP
p2 xref=45 size=720x480 H36=0.00 H37=0.22 → KEEP
p3 xref=48 size=2480x3508 H36=0.98 H37=0.95 → SKIP
p3 xref=49 size=989x499 H36=0.51 H37=0.51 → KEEP
p3 xref=50 size=640x480 H36=0.00 H37=0.14 → KEEP
p3 xref=51 size=976x732 H36=0.00 H37=0.14 → KEEP
p4 xref=54 size=2480x3508 H36=0.98 H37=0.95 → SKIP
p4 xref=70 size=240x119 H36=0.55 H37=0.20 → KEEP
p4 xref=71 size=853x480 H36=0.00 H37=0.15 → KEEP
p4 xref=72 size=480x640 H36=0.00 H37=0.16 → KEEP
p4 xref=73 size=1024x684 H36=0.00 H37=0.20 → KEEP
p5 xref=76 size=2480x3508 H36=0.98 H37=0.95 → SKIP
p5 xref=77 size=240x119 H36=0.55 H37=0.20 → KEEP
p5 xref=78 size=327x505 H36=0.89 H37=0.89 → SKIP
p5 xref=79 size=509x336 H36=0.00 H37=0.15 → KEEP
p5 xref=80 size=763x572 H36=0.00 H37=0.22 → KEEP
p6 xref=83 size=2480x3508 H36=0.98 H37=0.95 → SKIP
p6 xref=84 size=405x519 H36=0.91 H37=0.90 → SKIP
p6 xref=